# PolyChain: Polymer Property Prediction
**AISEHack 2.0 - Polymer Property Prediction**

Run all cells in order. Autosave: completed folds are skipped on re-run.
Supports resume after disconnection.

## Pipeline
1. Install & setup
2. Split data by target_type -> tg/egc
3. Build features (Morgan 2048 + MACCS 167 + AtomPair 2048 + Torsion 2048 + 200+ RDKit descriptors + 20+ polymer features)
4. Generate CV splits (5-fold scaffold-based)
5. Train models (ridge/xgb/lgb/catboost/rf/mlp) per target per fold
6. Blend ensemble per target (weighted by inverse RMSE)
7. Merge per-target submissions -> final submission.csv
8. Visualizations: target distributions, feature importance, OOF scatter, model comparison

In [ ]:
# Cell 1: Install dependencies
import sys, os, json, pickle, subprocess, shutil, time, warnings, zipfile
from pathlib import Path

print("Python:", sys.version[:6])
print("Executable:", sys.executable)

# Check available package managers
conda_candidates = [
    "/opt/conda/bin/conda",
    "/usr/local/conda/bin/conda",
    shutil.which("conda"),
]
conda = None
for c in conda_candidates:
    if c and Path(c).exists():
        conda = c
        print(f"Found conda: {c}")
        break
else:
    print("conda not found, will use pip")

try:
    import rdkit
    print("rdkit already installed")
except ImportError:
    print("Installing rdkit...")
    if conda:
        r = subprocess.run(
            [conda, "install", "-c", "conda-forge", "rdkit", "-y", "-q"],
            capture_output=True, text=True, timeout=300
        )
        print(r.stdout[-300:] if r.stdout else "")
        print(r.stderr[-300:] if r.stderr else "")
    else:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "rdkit"],
            capture_output=True, text=True, timeout=120
        )
        print(r.stdout[-300:] if r.stdout else "")
        print(r.stderr[-300:] if r.stderr else "")
    import rdkit
    print("rdkit ready!")

try:
    import matplotlib
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "matplotlib", "seaborn"],
        timeout=120
    )

print("Dependencies ready")


In [ ]:
# Cell 2: Environment setup
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess, shutil, zipfile

KAGGLE = Path("/kaggle").exists()
COLAB = Path("/content").exists() and not KAGGLE

if KAGGLE:
    COMP_SRC = Path("/kaggle/input/competitions/aisehack-2-0")
    CODE_SRC = Path("/kaggle/input/datasets/shubhamkambli11/polymer-competition-pipeline")
    WORK_DIR = Path("/kaggle/working/polymer_competition")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    print("COMP_SRC exists:", COMP_SRC.exists())
    print("CODE_SRC exists:", CODE_SRC.exists())
    if CODE_SRC.exists():
        print("Copying code dataset...")
        for item in CODE_SRC.iterdir():
            dest = WORK_DIR / item.name
            if item.is_dir() and not dest.exists():
                shutil.copytree(item, dest, dirs_exist_ok=True)
            elif item.is_file():
                shutil.copy2(item, dest)
    if COMP_SRC.exists():
        (WORK_DIR / "data").mkdir(parents=True, exist_ok=True)
        shutil.copy(COMP_SRC / "train.csv", WORK_DIR / "data/train.csv")
        shutil.copy(COMP_SRC / "test.csv", WORK_DIR / "data/test.csv")
        print("Copied train/test from competition")
        # Copy entire code tree (dataset is already extracted)
        for item in CODE_SRC.iterdir():
            dest = WORK_DIR / item.name
            if item.is_dir() and not dest.exists():
                shutil.copytree(item, dest, dirs_exist_ok=True)
            elif item.is_file():
                shutil.copy2(item, dest)
    # Override dataset code with latest from GitHub (ensures fixes are applied)
    GIT_REPO = "https://github.com/NotShubham1112/Poly.git"
    GIT_DIR = WORK_DIR.parent / "Poly"
    import subprocess as _sp
    if not GIT_DIR.exists():
        _sp.run(["git", "clone", "--depth", "1",
                 "-b", "feat/competition-data-adaptation",
                 GIT_REPO, str(GIT_DIR)],
                capture_output=True, timeout=60)
    else:
        _sp.run(["git", "-C", str(GIT_DIR), "pull",
                 "origin", "feat/competition-data-adaptation"],
                capture_output=True, timeout=60)
    if GIT_DIR.exists():
        for _item in GIT_DIR.iterdir():
            _dest = WORK_DIR / _item.name
            if _item.is_dir() and _item.name not in ('.git', '__pycache__'):
                if _dest.exists():
                    shutil.rmtree(_dest)
                shutil.copytree(_item, _dest, dirs_exist_ok=True)
            elif _item.is_file() and _item.name.endswith(('.py', '.yaml', '.txt')):
                shutil.copy2(_item, _dest)
        print("Updated code from GitHub (feat/competition-data-adaptation)")

        sys.path.insert(0, str(WORK_DIR))
    os.chdir(WORK_DIR)
    print("WORK_DIR contents:", [p.name for p in WORK_DIR.iterdir() if not p.name.startswith(".")][:30])
    print("Has config.yaml:", Path("config.yaml").exists())
    print("Has data/train.csv:", Path("data/train.csv").exists())
elif COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/Poly/polymer_competition")
    os.chdir(WORK_DIR)
    # Override dataset code with latest from GitHub (ensures fixes are applied)
    GIT_REPO = "https://github.com/NotShubham1112/Poly.git"
    GIT_DIR = WORK_DIR.parent / "Poly"
    import subprocess as _sp
    if not GIT_DIR.exists():
        _sp.run(["git", "clone", "--depth", "1",
                 "-b", "feat/competition-data-adaptation",
                 GIT_REPO, str(GIT_DIR)],
                capture_output=True, timeout=60)
    else:
        _sp.run(["git", "-C", str(GIT_DIR), "pull",
                 "origin", "feat/competition-data-adaptation"],
                capture_output=True, timeout=60)
    if GIT_DIR.exists():
        for _item in GIT_DIR.iterdir():
            _dest = WORK_DIR / _item.name
            if _item.is_dir() and _item.name not in ('.git', '__pycache__'):
                if _dest.exists():
                    shutil.rmtree(_dest)
                shutil.copytree(_item, _dest, dirs_exist_ok=True)
            elif _item.is_file() and _item.name.endswith(('.py', '.yaml', '.txt')):
                shutil.copy2(_item, _dest)
        print("Updated code from GitHub (feat/competition-data-adaptation)")

        sys.path.insert(0, str(WORK_DIR))
    src = Path("/content/drive/MyDrive/PolyChain/data")
    if src.exists():
        shutil.copy(src / "train.csv", "data/train.csv")
        shutil.copy(src / "test.csv", "data/test.csv")
else:
    WORK_DIR = Path.cwd()
    os.chdir(WORK_DIR)

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
N_FOLDS = cfg["cv"]["n_folds"]
EXP_VER = cfg.get("experiment", {}).get("version", "v1")

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
env_str = "Kaggle" if KAGGLE else "Colab" if COLAB else "Local"
print(f"Environment: {env_str}")
print(f"Train: {len(train)} rows | Test: {len(test)} rows")
targets_dict = train["target_type"].value_counts().to_dict()
print(f"Targets: {targets_dict}")


In [ ]:
# Cell 3: EDA - Target distributions
TARGETS = ["tg", "egc"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for i, t in enumerate(TARGETS):
    t_df = train[train["target_type"] == t]
    axes[i].hist(t_df["target"], bins=50, edgecolor="white", alpha=0.7)
    axes[i].set_title(f"{t.upper()} Distribution (n={len(t_df)})")
    axes[i].set_xlabel("Target value")
    axes[i].set_ylabel("Count")
plt.tight_layout()
os.makedirs("reports/plots", exist_ok=True)
plt.savefig("reports/plots/target_distributions.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Tg: mean={train[train['target_type']=='tg']['target'].mean():.1f}, std={train[train['target_type']=='tg']['target'].std():.1f}")
print(f"Egc: mean={train[train['target_type']=='egc']['target'].mean():.2f}, std={train[train['target_type']=='egc']['target'].std():.2f}")

In [ ]:
# Cell 4: Split data by target type
from data.split_by_target import split_by_target
split_by_target("data/train.csv", "data/test.csv", "data/")

for t in TARGETS:
    t_train = pd.read_csv(f"data/{t}/train.csv")
    t_test = pd.read_csv(f"data/{t}/test.csv")
    print(f"{t}: {len(t_train)} train, {len(t_test)} test")

In [ ]:
# Cell 5: Build features (all types)"
from features import descriptors as _desc_mod
_desc_mod.DESCRIPTOR_NAMES = [n for n in _desc_mod.DESCRIPTOR_NAMES if "EState" not in n]
print(f"Using {len(_desc_mod.DESCRIPTOR_NAMES)} descriptors (EState excluded)")
from features.build_features import build_features
build_features("config.yaml")

train_feat = pd.read_parquet("data/processed/features_train.parquet")
test_feat = pd.read_parquet("data/processed/features_test.parquet")

# Show feature breakdown
morgan_cols = [c for c in train_feat.columns if c.startswith("morgan_")]
maccs_cols = [c for c in train_feat.columns if c.startswith("maccs_")]
atom_pair_cols = [c for c in train_feat.columns if c.startswith("atom_pair_")]
torsion_cols = [c for c in train_feat.columns if c.startswith("torsion_")]
desc_cols = [c for c in train_feat.columns if c not in (["SMILES", "id", "canon_smiles"]
    + morgan_cols + maccs_cols + atom_pair_cols + torsion_cols)]
print(f"Feature breakdown:")
print(f"  Morgan (ECFP6): {len(morgan_cols)}")
print(f"  MACCS keys:     {len(maccs_cols)}")
print(f"  Atom pair:      {len(atom_pair_cols)}")
print(f"  Topol. torsion: {len(torsion_cols)}")
print(f"  RDKit desc + poly: {len(desc_cols)}")
print(f"  Total features: {train_feat.shape[1] - 3}")
print(f"  Train: {train_feat.shape}, Test: {test_feat.shape}")

In [ ]:
# Cell 6: Generate CV splits
from data.generate_splits import generate_splits

for target in TARGETS:
    generate_splits(
        f"data/{target}/train.csv",
        f"data/splits_{target}.pkl",
        n_folds=N_FOLDS, smiles_col="smiles", target_col="target",
    )

In [ ]:
# Cell 7: Train all models with autosave/resume
# Each completed fold saves a .pkl to predictions/ ; re-running skips existing
MODELS = ["mlp", "ridge", "xgb", "lgb", "catboost", "rf"]
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))

pred_dir = Path("predictions")
pred_dir.mkdir(exist_ok=True)

for target in TARGETS:
    print(f"\n{'='*60}")
    print(f"  TARGET: {target}")
    print(f"{'='*60}")
    for model_type in MODELS:
        for fold in range(N_FOLDS):
            pred_file = pred_dir / f"{EXP_VER}_{target}_{model_type}_fold{fold}.pkl"
            if pred_file.exists():
                print(f"  SKIP (exists): {pred_file.name}")
                continue

            cmd = [
                sys.executable, "-m", "training.train",
                "--model_type", model_type,
                "--fold", str(fold),
                "--config", "config.yaml",
                "--target", target,
            ]
            t0 = time.time()
            result = subprocess.run(cmd, capture_output=True, text=True)
            elapsed = time.time() - t0
            if result.returncode == 0:
                for line in result.stdout.split("\n"):
                    if "rmse" in line and "r2" in line:
                        print(f"  {model_type} fold {fold}: {line.strip()} ({elapsed:.0f}s)")
                        break
            else:
                stderr_short = result.stderr.strip().split("\n")[-1] if result.stderr else ""
                print(f"  {model_type} fold {fold}: FAILED ({elapsed:.0f}s) - {stderr_short[:80]}")

In [ ]:
# Cell 8: Build ensemble per target (weighted blend by inverse RMSE)
for target in TARGETS:
    result = subprocess.run([
        sys.executable, "-m", "ensemble.build_ensemble",
        "--config", "config.yaml", "--target", target,
    ], capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print(f"ERROR: {result.stderr}")

In [ ]:
# Cell 9: Merge per-target submissions -> final submission.csv
result = subprocess.run([
    sys.executable, "-m", "data.merge_submissions",
    "--config", "config.yaml",
], capture_output=True, text=True)
print(result.stdout if result.stdout else "")
if result.returncode != 0 and result.stderr:
    print(f"ERROR: {result.stderr}")

In [ ]:
# Cell 10: Visualizations - model comparison per target
report_dir = Path("reports/plots")
report_dir.mkdir(parents=True, exist_ok=True)

# Load all prediction files and aggregate metrics
all_metrics = []
for pkl_file in pred_dir.glob(f"{EXP_VER}_*_*.pkl"):
    if pkl_file.stem.endswith("_test"):
        continue
    with open(pkl_file, "rb") as f:
        d = pickle.load(f)
    all_metrics.append({
        "target": d.get("target", "?"),
        "model": d.get("model_type", "?"),
        "fold": d.get("fold", 0),
        "r2": d.get("metrics", {}).get("r2", None),
        "rmse": d.get("metrics", {}).get("rmse", None),
        "mae": d.get("metrics", {}).get("mae", None),
    })

if all_metrics:
    df_metrics = pd.DataFrame(all_metrics)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, t in enumerate(TARGETS):
        t_df = df_metrics[df_metrics["target"] == t]
        if len(t_df) == 0:
            continue
        agg = t_df.groupby("model")[["r2", "rmse"]].agg(["mean", "std"])
        models = agg.index.tolist()
        r2_means = agg["r2"]["mean"].values
        r2_stds = agg["r2"]["std"].values
        axes[i].barh(models, r2_means, xerr=r2_stds, capsize=3)
        axes[i].axvline(0, color="gray", linestyle="-", alpha=0.3)
        axes[i].set_title(f"{t.upper()} - RÂ² by Model (mean across folds)")
        axes[i].set_xlabel("RÂ²")
    plt.tight_layout()
    plt.savefig(report_dir / "model_comparison.png", dpi=100, bbox_inches="tight")
    plt.show()

    # Print best model per target
    print("\nBest model per target (mean RÂ²):")
    for t in TARGETS:
        t_df = df_metrics[df_metrics["target"] == t]
        if len(t_df):
            best = t_df.groupby("model")["r2"].mean().idxmax()
            best_r2 = t_df.groupby("model")["r2"].mean().max()
            print(f"  {t}: {best} (RÂ²={best_r2:.4f})")
else:
    print("No prediction files found to plot.")

In [ ]:
# Cell 11: OOF scatter plots (predicted vs actual)
os.makedirs("reports/plots", exist_ok=True)
if all_metrics:
    for target in TARGETS:
        # Collect all OOF predictions for this target
        all_preds, all_y = [], []
        for pkl_file in sorted(pred_dir.glob(f"{EXP_VER}_{target}_*.pkl")):
            if pkl_file.stem.endswith("_test"):
                continue
            with open(pkl_file, "rb") as f:
                d = pickle.load(f)
            all_preds.extend(d["pred"][:500])  # sample 500 for readability
            all_y.extend(d["y"][:500])
        
        if all_preds:
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.scatter(all_y, all_preds, alpha=0.4, s=10)
            lims = [min(all_y + all_preds), max(all_y + all_preds)]
            ax.plot(lims, lims, "r--", alpha=0.5, label="Ideal")
            ax.set_xlabel("Actual")
            ax.set_ylabel("Predicted")
            ax.set_title(f"{target.upper()} - OOF Predictions")
            ax.legend()
            plt.tight_layout()
            plt.savefig(report_dir / f"oof_scatter_{target}.png", dpi=100, bbox_inches="tight")
            plt.show()
else:
    print("No predictions for OOF scatter plots.")

In [ ]:
# Cell 12: Ensemble weights visualization
os.makedirs("reports/plots", exist_ok=True)
for target in TARGETS:
    weight_file = Path(f"ensembles/{EXP_VER}_{target}_weights.json")
    if weight_file.exists():
        with open(weight_file) as f:
            w_data = json.load(f)
        weights = w_data["weights"]
        fig, ax = plt.subplots(figsize=(8, 3))
        models_w = list(weights.keys())
        vals = list(weights.values())
        colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(models_w)))
        ax.barh(models_w, vals, color=colors)
        ax.set_title(f"{target.upper()} - Ensemble Weights (CV RMSE={w_data['cv_score']:.2f})")
        ax.set_xlabel("Weight")
        for i, v in enumerate(vals):
            ax.text(v + 0.01, i, f"{v:.3f}", va="center")
        plt.tight_layout()
        plt.savefig(report_dir / f"weights_{target}.png", dpi=100, bbox_inches="tight")
        plt.show()
    else:
        print(f"No weights file for {target}")

In [ ]:
# Cell 13: Submission preview & export
sub = pd.read_csv("outputs/submissions/submission.csv")
print(f"Submission: {len(sub)} rows, columns: {list(sub.columns)}")
print(f"ID range: {sub['id'].min()} - {sub['id'].max()}")
print(f"\nFirst 5:")
print(sub.head())
print(f"\nLast 5:")
print(sub.tail())

if KAGGLE:
    shutil.copy("outputs/submissions/submission.csv", "/kaggle/working/submission.csv")
    print("\nCopied to /kaggle/working/submission.csv")

In [ ]:
# Cell 14: Full summary
print("="*60)
print("  PIPELINE COMPLETE")
print("="*60)
print(f"  Models: {len(MODELS)} x {len(TARGETS)} targets x {N_FOLDS} folds")
print(f"  Saved: outputs/submissions/submission.csv")
print(f"  Plots: reports/plots/")
print(f"  Log:   experiments/manifest.json")
print(f"  Weight ensembles/: ensembes/")
print("="*60)

manifest_path = Path("experiments/manifest.json")
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    scores = [r for r in manifest if r.get("score") is not None]
    if scores:
        df_scores = pd.DataFrame(scores)
        print(f"\n  Best RÂ² per target:")
        for target in TARGETS:
            t_scores = df_scores[df_scores["target"] == target]
            if len(t_scores):
                best = t_scores.loc[t_scores["score"].idxmax()]
                print(f"    {target}: {best['model']} fold {best['fold']} RÂ²={best['score']:.4f}")